# VMamba-T: lime

Explain all **45 original bandwidth validation images**, including mistakes.
Restore the existing best-validation-accuracy checkpoint: archived result
**44/45**. No training is performed. Classes: **0 Low, 1 High bandwidth**.
Use [START_HERE.md](START_HERE.md) and notebook 00 to set up the lab computer.
These validation images were used in checkpoint selection; they are not an
independent test set.


In [1]:
from pathlib import Path
import sys
import numpy as np
import torch

# Usually leave this as None. Set a path only if Jupyter starts elsewhere.
PACKAGE_ROOT = None
if PACKAGE_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p/'package_integrity.json').is_file()
                 and (p/'xai_code').is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError('Start Jupyter inside the extracted package, or set PACKAGE_ROOT.')
else:
    ROOT = Path(PACKAGE_ROOT).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from xai_code.models import build_model, Predictor
from xai_code.runtime import prepare, audit_predictions, verify_package, environment, samples
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Package:', ROOT)
print('Device:', DEVICE)


Package: /home/murat/Desktop/original_bandwidth_explainability_lab
Device: cuda


## Configuration

In [2]:
MODEL_NAME = 'vmamba'
METHOD = 'lime'
BATCH_SIZE = 4  # Reduce this if GPU memory is insufficient.
ALLOW_PREDICTION_DIFFERENCES = False
SETTINGS = {
    'seed': 42,
    'batch_size': BATCH_SIZE,
    'allow_prediction_differences': ALLOW_PREDICTION_DIFFERENCES,
}

SETTINGS.update({
    'n_segments': 100,
    'compactness': 10,
    'num_samples': 1000,
    'hide_color': 255,
    'kernel_width': 0.25,
    'display_features': None,  # None = all positive segments; or set a top-k.
    'segmentation_resolution': 'native, common across models',
    'feature_selection': 'none',
    'distance_metric': 'cosine',
})


## Restore the fixed checkpoint and verify all predictions

In [3]:
OUTPUT = prepare(ROOT, MODEL_NAME, METHOD, SETTINGS, DEVICE)
model = build_model(ROOT, MODEL_NAME, DEVICE)
predictor = Predictor(model, MODEL_NAME, DEVICE, batch_size=BATCH_SIZE)
ROWS, AUDIT = audit_predictions(ROOT, MODEL_NAME, predictor, OUTPUT,
    allow_prediction_differences=ALLOW_PREDICTION_DIFFERENCES)
print('Ready to explain ALL', len(ROWS), 'validation images.')


Integrity OK: 84 fixed files; notebook existence checked separately
Outputs: /home/murat/Desktop/original_bandwidth_explainability_lab/outputs/vmamba/lime/26e7bb04efbc3d97


/home/murat/Desktop/original_bandwidth_explainability_lab/xai_code/vmamba_official.py:853: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/home/murat/Desktop/original_bandwidth_explainability_lab/xai_code/vmamba_official.py:868: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/home/murat/Desktop/training_models/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Validation audit: 44/45; 0 changed predictions
Ready to explain ALL 45 validation images.


## LIME implementation

The following cells contain the actual explainer and per-image execution code.
Reported paper settings: SLIC ≈100 segments, compactness 10, connectivity,
1,000 perturbations, white masking, positive evidence for the predicted class.
Shared native-resolution segmentation and the recorded surrogate settings
are explicit choices for comparing our four classifiers. Every perturbation
passes through the full original preprocessing, backbone and head.


In [4]:
"""Paper-style white masking / SLIC LIME, extended to the four saved models."""
from pathlib import Path
import numpy as np
from PIL import Image
from lime import lime_image
from skimage.segmentation import slic, mark_boundaries
from xai_code.runtime import read_rgb, image_seed, write_json, save_figure, summarize


In [5]:
def explain_one(rgb, predictor, predicted_class, seed, settings):
    # Segment the original native-resolution image ONCE. All models use this
    # same coordinate system; their original resizing happens inside Predictor.
    # This cross-model comparison choice is recorded, rather than described
    # as a pixel-for-pixel reproduction of the authors' CNN explanation images.
    segments = slic(rgb.astype(np.float32)/255.0,
        n_segments=settings['n_segments'], compactness=settings['compactness'],
        enforce_connectivity=True, start_label=0, channel_axis=-1, convert2lab=True)
    ids = np.unique(segments)
    if not np.array_equal(ids, np.arange(len(ids))):
        raise ValueError('LIME requires contiguous segment IDs beginning at zero')
    explainer = lime_image.LimeImageExplainer(
        random_state=seed, kernel_width=settings['kernel_width'], feature_selection='none')
    explanation = explainer.explain_instance(
        rgb, classifier_fn=predictor, labels=(predicted_class,), top_labels=None,
        hide_color=settings['hide_color'], num_samples=settings['num_samples'],
        num_features=len(ids), batch_size=settings['batch_size'],
        distance_metric='cosine', segmentation_fn=lambda image: segments,
        random_seed=seed)
    limit = settings['display_features'] or len(ids)
    _, positive_mask = explanation.get_image_and_mask(
        predicted_class, positive_only=True, hide_rest=False,
        num_features=limit, min_weight=0.0)
    coefficients = np.zeros(len(ids), dtype=np.float32)
    for segment_id, weight in explanation.local_exp[predicted_class]:
        coefficients[int(segment_id)] = float(weight)
    # Released LIME 0.2.0.1 stores score/local_pred as scalars/arrays; newer
    # source stores dictionaries. Both forms are supported for one target class.
    def scalar(value):
        if isinstance(value, dict):
            value = value[predicted_class]
        number = float(np.asarray(value).reshape(-1)[0])
        return number if np.isfinite(number) else None
    details = {'segment_count': len(ids), 'seed': seed,
        'local_weighted_r2': scalar(explanation.score),
        'local_surrogate_prediction': scalar(explanation.local_pred),
        'surrogate_intercept': scalar(explanation.intercept),
        'positive_segment_count': int(sum(weight > 0 for weight in coefficients)),
        'selected_positive_segment_count': int(len(np.unique(segments[positive_mask > 0]))),
        'positive_pixel_fraction': float((positive_mask > 0).mean())}
    return segments, coefficients, positive_mask.astype(np.uint8), details


In [6]:
def run_lime(root, rows, audit, predictor, output, settings):
    from matplotlib import pyplot as plt
    for index, row in enumerate(rows, start=1):
        destination = output/Path(row['file']).stem
        if all((destination/name).is_file() for name in ['done.json', 'maps.npz', 'explanation.png', 'coefficients.json', 'segments.png']):
            print(f"[{index}/45] Already complete: {row['file']}")
            continue
        destination.mkdir(parents=True, exist_ok=True)
        rgb = read_rgb(root, row)
        record = audit[row['file']]
        target = record['predicted_label']
        print(f"[{index}/45] LIME: {row['file']}; predicted {target}; correct={record['correct']}")
        segments, coefficients, mask, details = explain_one(
            rgb, predictor, target, image_seed(row['file'], settings['seed']), settings)
        # Coefficients are in probability units, not a proof of physical causality.
        np.savez_compressed(destination/'maps.npz', segments=segments,
                            coefficients=coefficients, signed_weights=coefficients[segments],
                            positive_mask=mask)
        Image.fromarray(rgb).save(destination/'original.png')
        plt.imsave(destination/'segments.png', mark_boundaries(rgb/255.0, segments))
        write_json(destination/'coefficients.json', {
            'explained_class': target, 'segments': [
                {'segment_id': int(i), 'coefficient': float(weight)}
                for i, weight in enumerate(coefficients)]})
        save_figure(rgb, mask, f"LIME: {'High' if target else 'Low'}; correct={record['correct']}",
                    destination/'explanation.png', binary=True)
        write_json(destination/'done.json', {**record, **details, 'method': 'lime',
                    'explained_class': target, 'native_width': rgb.shape[1], 'native_height': rgb.shape[0]})
    return summarize(root, rows, audit, output, method='lime')


## Run LIME for all 45 images; resume completed images

In [ ]:
summaries = run_lime(ROOT, ROWS, AUDIT, predictor, OUTPUT, SETTINGS)


[1/45] LIME: M0_log-20241114-212937.bmp; predicted 0; correct=True


  0%|          | 0/1000 [00:00<?, ?it/s]

[2/45] LIME: M10_log-20241114-212937.bmp; predicted 1; correct=True


  0%|          | 0/1000 [00:00<?, ?it/s]

[3/45] LIME: M11_log-20241114-212937.bmp; predicted 1; correct=True


...

  0%|          | 0/1000 [00:00<?, ?it/s]

[22/45] LIME: M238_log-20241114-212937.bmp; predicted 1; correct=True


  0%|          | 0/1000 [00:00<?, ?it/s]

[23/45] LIME: M240_log-20241114-212937.bmp; predicted 0; correct=True


  0%|          | 0/1000 [00:00<?, ?it/s]

## Inspect saved examples, including errors

In [ ]:
from IPython.display import display, Image as DisplayImage
print('Saved results:', OUTPUT)
error_names = [row['file'] for row in ROWS if not AUDIT[row['file']]['correct']]
example_names = list(dict.fromkeys(error_names + [ROWS[0]['file']]))
for filename in example_names:
    print(filename, AUDIT[filename])
    figure = OUTPUT/Path(filename).stem/'explanation.png'
    if figure.is_file():
        display(DisplayImage(filename=str(figure)))


Outputs include per-image raw maps and metadata, a prediction audit,
summary CSV and class means grouped by predicted class, including mistakes.
Read the local fit scores for LIME. Heatmaps alone do not establish physical
causality. The earlier mean-brightness diagnostic is included in `reference`.

References: [photonic LIME paper](https://advanced.onlinelibrary.wiley.com/doi/10.1002/adpr.202500284),
[LIME](https://github.com/marcotcr/lime),
[attention rollout](https://arxiv.org/abs/2005.00928),
[gradient-weighted rollout variant](https://github.com/jacobgil/vit-explain/blob/main/vit_grad_rollout.py).
